# AEGIS: Advanced Structural Inspection ML Pipeline
### Unified Lifecycle: Data Audit -> Optimized Training -> Visual Analytics -> Backend Handover

**Project AEGIS** (Advanced Engineering & Geospatial Inspection System) requires a high-precision crack detection model. This notebook provides an automated pipeline to train, evaluate, and synchronize YOLOv8 weights with the project backend.

## Stage 1: Control Room (Setup & Hardware Check)
We initialize the environment, mount storage, and verify GPU acceleration.

In [ ]:
# 1.1 Environment Setup
!pip install -q ultralytics roboflow seaborn pandas matplotlib

import os
import yaml
import shutil
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from ultralytics import YOLO
from roboflow import Roboflow
from google.colab import drive, userdata # Secure key handling

sns.set_theme(style="whitegrid")

# 1.2 Drive Mounting for Persistence
if 'COLAB_GPU' in os.environ:
    drive.mount('/content/drive')
    WEIGHTS_STORAGE = "/content/drive/MyDrive/AEGIS_WORKSPACE/v4_production"
    os.makedirs(WEIGHTS_STORAGE, exist_ok=True)
    print("✅ Storage Linked: Google Drive")
else:
    print("⚠️ Not in Colab environment. Local storage will be used.")

## Stage 2: Data Ingestion & Audit
Checking data distribution and quality before training. This ensures the model isn't biased towards a single crack type.

In [ ]:
# 2.1 Secure Dataset Download
# Note: Add 'ROBOFLOW_API_KEY' to your Colab Secrets (Key Icon on left)
API_KEY = userdata.get('ROBOFLOW_API_KEY')
rf = Roboflow(api_key=API_KEY)
project = rf.workspace("university-bswxt").project("crack-bphdr")
dataset = project.version(2).download("yolov8")

# 2.2 Graphical Presentation: Data Audit
with open(f"{dataset.location}/data.yaml", 'r') as f:
    info = yaml.safe_load(f)
     
plt.figure(figsize=(10, 5))
# Using total image count as a proxy for distribution audit
train_images = len(os.listdir(f"{dataset.location}/train/images"))
val_images = len(os.listdir(f"{dataset.location}/valid/images"))

plt.bar(['Training Set', 'Validation Set'], [train_images, val_images], color=['#1f77b4', '#ff7f0e'])
plt.title("AEGIS Inspection Data: Volume Audit", fontsize=15)
plt.ylabel("Image Count")
plt.show()

## Stage 3: Optimized Training
We use **YOLOv8 Small (yolov8s.pt)** for better structural feature extraction compared to the Nano version. We also deploy **Mosaic** and **Mixup** augmentations to handle varying drone light conditions.

In [ ]:
# 3.1 Model Initialization
model = YOLO('yolov8s.pt') # 's' provides higher spatial precision for cracks

# 3.2 Production training loop
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100,
    imgsz=640,
    patience=20,     # Early stopping for efficiency
    batch=-1,        # Auto-calculate batch size for GPU
    optimizer='AdamW', # Stable weight decay for fine textures
    augment=True,    # Enabled for drone-flight robustness
    mosaic=1.0,      # High mosaic for complex backgrounds (bridges/roads)
    mixup=0.1, 
    name='aegis_production_v4'
)

## Stage 4: Qualitative Performance Hub
Graphical presentation of the model's accuracy and learning curves.

In [ ]:
# 4.1 Training Metrics Extraction
results_df = pd.read_csv('runs/detect/aegis_production_v4/results.csv')
results_df.columns = results_df.columns.str.strip()

plt.figure(figsize=(15, 6))

# Plotting mAP Accuracy Curve
plt.subplot(1, 2, 1)
sns.lineplot(data=results_df, x='epoch', y='metrics/mAP50(B)', marker='o', color='green')
plt.fill_between(results_df['epoch'], results_df['metrics/mAP50(B)'], alpha=0.2, color='green')
plt.title("AEGIS Precision Curve (mAP@50)", fontsize=14)

# Plotting Loss Decay
plt.subplot(1, 2, 2)
sns.lineplot(data=results_df, x='epoch', y='train/box_loss', label='Box Loss', color='red')
sns.lineplot(data=results_df, x='epoch', y='train/cls_loss', label='Cls Loss', color='blue')
plt.title("Training Loss Decay", fontsize=14)
plt.show()

## Stage 5: Inference HUD & Comparative Test
Testing the model on sample structural images to visualize performance in the field.

In [ ]:
import cv2
from google.colab.patches import cv2_imshow

# 5.1 Test run on Validation sample
test_images = list(os.listdir(f"{dataset.location}/valid/images"))[:3]

print("🔍 Visualizing Model Predictions vs Field Standards")
for img_file in test_images:
    img_path = f"{dataset.location}/valid/images/{img_file}"
    pred = model.predict(img_path, conf=0.3)
    result_img = pred[0].plot() # High-res plotted result
    
    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title(f"AEGIS HUD: {img_file}")
    plt.show()

## Stage 6: Project Synchronization (Backend Handover)
Automatically exporting weights and synchronizing with the AEGIS FastAPI project folder.

In [ ]:
# 6.1 Automated Handover Script
PROJECT_ROOT = "./"
BACKEND_WEIGHTS = os.path.join(PROJECT_ROOT, "backend/models/crack_detection/weights/best.pt")
TRAINED_WEIGHTS = "runs/detect/aegis_production_v4/weights/best.pt"

# Ensure directories exist for the project
os.makedirs(os.path.dirname(BACKEND_WEIGHTS), exist_ok=True)

if os.path.exists(TRAINED_WEIGHTS):
    # 1. Update Project Backend
    shutil.copy(TRAINED_WEIGHTS, BACKEND_WEIGHTS)
    
    # 2. Export and Backup to Drive
    if 'COLAB_GPU' in os.environ:
        shutil.copy(TRAINED_WEIGHTS, f"{WEIGHTS_STORAGE}/aegis_best_v4.pt")
        print(f"⭐ Backup successful at: {WEIGHTS_STORAGE}")
    
    # 3. Export to ONNX for production speed boosting
    onnx_path = model.export(format='onnx', opset=12)
    
    print("\n🚀 PIPELINE COMPLETE")
    print(f"✅ Backend weights updated at: {BACKEND_WEIGHTS}")
    print("✅ Project is now synchronized with new model weights.")
else:
    print("❌ FAIL: Weights not found. Double check training logs.")